# MS MARCO RARS-v3 Oracle-First Feasibility

## tl;dr

This notebook runs the frozen RARS-v3 oracle-first gate at implementation commit \`00cc09426ce98fd60d0d800c80fab0c8fc890f03\`. It measures whether a non-deployable query-label oracle has enough comparator-relative Recall@10 headroom at 0/320/640/1280 accessed bytes per query to justify a later static-storage oracle.

This is development-only evidence on historical v2 development queries. It is not independent confirmation, not a learned method, and not a persistent-storage result.

## Context & Methods

Key assumptions are frozen before outcomes are opened:

- candidates are exact deterministic subsets of the frozen v2.2 Top-100 arrays;
- the v3 candidate builder does not open qrels, import Faiss, rerun retrieval, or read parent label payload bytes;
- one label-free uncentered rank-32 progressive SVD representation is fitted on the design role;
- the best of five registered 640-byte comparators is selected on design only;
- audit labels are materialized only inside the evaluator after a durable design freeze is recursively verified;
- the formal outcome is one of INVALID, KILL_NO_SCORE_HEADROOM, STOP_NO_HEADROOM, or GO_TO_STATIC_STORAGE_ORACLE.

A GO result authorizes only a separately frozen static serialized-storage oracle study. It does not authorize training, QAT, deployment, or a storage claim.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

EXPERIMENT_PYTHON = sys.executable
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q',
    'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9',
], check=True)
NUMPY_TARGET = Path('/content/rars-v3-numpy126')
if NUMPY_TARGET.exists():
    shutil.rmtree(NUMPY_TARGET)
NUMPY_TARGET.mkdir(parents=True)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q', '--no-deps',
    '--target', str(NUMPY_TARGET), 'numpy==1.26.4',
], check=True)
EXPERIMENT_ENV = os.environ.copy()
EXPERIMENT_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [
    str(NUMPY_TARGET), EXPERIMENT_ENV.get('PYTHONPATH', ''),
]))
installed_numpy_version = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c', 'import numpy; print(numpy.__version__)',
], text=True, env=EXPERIMENT_ENV).strip()
assert installed_numpy_version == '1.26.4', installed_numpy_version
installed_numpy_path = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c', 'import numpy; print(numpy.__file__)',
], text=True, env=EXPERIMENT_ENV).strip()
assert Path(installed_numpy_path).resolve().is_relative_to(NUMPY_TARGET.resolve())
print('Colab host-kernel NumPy (not used by experiments):',
      getattr(sys.modules.get('numpy'), '__version__', 'not-loaded'))
print('Fresh experiment-subprocess NumPy:', installed_numpy_version)

from google.colab import drive
drive.mount('/content/drive')

import hashlib, json

TRAINING_COMMIT = 'bb9b106e69b9a453756fd800665f701614ce67b3'
V3_IMPLEMENTATION_COMMIT = '00cc09426ce98fd60d0d800c80fab0c8fc890f03'
REPO_URL = 'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git'
TRAIN_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2_2')
V3_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v3_oracle')

PARENT_WORK = Path('/content') / f'rars-v2.2-{TRAINING_COMMIT[:12]}'
V3_WORK = Path('/content') / f'rars-v3-{V3_IMPLEMENTATION_COMMIT[:12]}'
for local_work in (PARENT_WORK, V3_WORK):
    if local_work.exists():
        shutil.rmtree(local_work)
    local_work.mkdir(parents=True)
PARENT_BUNDLES = PARENT_WORK / 'bundles'
PARENT_CANDIDATE_CACHE = PARENT_WORK / 'candidate-cache'
V3_BUNDLES = V3_WORK / 'bundles'

DRIVE = Path('/content/drive/MyDrive/rag-pq-checkpoints')
CACHE = DRIVE / 'msmarco_basis_gate0_cache'
CLEAN = DRIVE / 'rars_clean_split_v1'
PCA = DRIVE / 'rars_pca_comparator_v1'
INDEX = DRIVE / 'msmarco_1m_pq_residual_gate3/frozen_ivfpq_m32_nlist512.index'
OUTPUT = DRIVE / 'rars-v3-oracle-first' / V3_IMPLEMENTATION_COMMIT[:12]

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def verify_record(path, record):
    path = Path(path)
    assert path.is_file(), path
    assert path.stat().st_size == int(record['bytes']), path
    assert sha256_file(path) == record['sha256'], path

EXPERIMENT_PROBE_MARKER = 'RARS_V3_EXPERIMENT_ENV='
EXPERIMENT_PROBE = r'''
import json, sys
import faiss
import numpy as np
print('RARS_V3_EXPERIMENT_ENV=' + json.dumps({
    'python_version': '.'.join(map(str, sys.version_info[:3])),
    'python_full': sys.version,
    'python_executable': sys.executable,
    'numpy_version': np.__version__,
    'numpy_module_path': np.__file__,
    'faiss_version': str(getattr(faiss, '__version__', 'UNKNOWN')),
}, allow_nan=False))
'''

def probe_experiment_environment():
    completed = subprocess.run(
        [EXPERIMENT_PYTHON, '-c', EXPERIMENT_PROBE],
        text=True, capture_output=True, check=False, env=EXPERIMENT_ENV,
    )
    if completed.returncode != 0:
        print(completed.stderr)
        raise subprocess.CalledProcessError(
            completed.returncode, completed.args,
            output=completed.stdout, stderr=completed.stderr,
        )
    payloads = [
        line[len(EXPERIMENT_PROBE_MARKER):]
        for line in completed.stdout.splitlines()
        if line.startswith(EXPERIMENT_PROBE_MARKER)
    ]
    assert len(payloads) == 1, completed.stdout
    return json.loads(payloads[0])


In [ ]:
def clone_exact(destination, commit):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    head = subprocess.check_output(
        ['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True
    ).strip()
    dirty = subprocess.check_output(
        ['git', '-C', str(destination), 'status', '--porcelain'], text=True
    ).strip()
    assert head == commit, (head, commit)
    assert not dirty, dirty

clone_exact(TRAIN_REPO, TRAINING_COMMIT)
clone_exact(V3_REPO, V3_IMPLEMENTATION_COMMIT)

PROTOCOL_PATH = V3_REPO / 'protocols/rars_v3_oracle_first_feasibility_v1.json'
protocol = json.loads(PROTOCOL_PATH.read_text())
assert protocol['status'] == 'FROZEN_BEFORE_FIRST_ORACLE_RUN'
assert protocol['method_revision_allowed'] is False
assert protocol['outcome_informed_revision_allowed'] is False
assert protocol['parent_lineage']['parent_training_commit'] == TRAINING_COMMIT
assert protocol['parent_lineage']['closed_v2_2_commit'] == (
    '32291e2aed75a99999e83e5d168b1133b70cc866'
)

subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v2_2_core.py',
    'tests/test_freeze_rars_v2_2_inner_bundles.py',
    'tests/test_build_msmarco_rars_v2_boundary_bundles.py',
], cwd=TRAIN_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v3_oracle_core.py',
    'tests/test_build_msmarco_rars_v3_oracle_bundles.py',
    'tests/test_materialize_rars_v3_role_labels.py',
    'tests/test_evaluate_rars_v3_oracle_first_feasibility.py',
    'tests/test_rars_v3_oracle_protocol_contract.py',
], cwd=V3_REPO, check=True, env=EXPERIMENT_ENV)
print('Exact parent commit:', TRAINING_COMMIT)
print('Exact v3 implementation commit:', V3_IMPLEMENTATION_COMMIT)
print('Frozen v3 protocol SHA-256:', sha256_file(PROTOCOL_PATH))


In [ ]:
required = [
    CACHE / 'embeddings.fp16.memmap',
    CACHE / 'doc_ids.int64.memmap',
    CACHE / 'query_vectors.fp32.npy',
    CACHE / 'qrels_subset.json',
    INDEX,
    PCA / 'bases/pca_unweighted_rank16.float32.npy',
    PCA / 'sidecars/scales_pca_rank16.float32.npy',
    PCA / 'sidecars/codes_pca_rank16.int8.memmap',
    CLEAN / 'selected_config.json',
    CLEAN / 'bases/score_error_weighted_rank16.npy',
    CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy',
    CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, {'missing_artifacts': missing}
assert shutil.disk_usage('/content').free >= 4_000_000_000, 'Need 4 GB local disk'

current_environment = probe_experiment_environment()
contract = protocol['execution_environment_contract']
assert current_environment['python_version'] == contract['python_version']
assert current_environment['numpy_version'] == contract['numpy_version']
assert Path(current_environment['numpy_module_path']).resolve().is_relative_to(
    NUMPY_TARGET.resolve()
)
assert current_environment['faiss_version'] != 'UNKNOWN'
print(json.dumps(current_environment, indent=2))


## Data

The only qrels access in this notebook occurs in the inherited, exact v2.2 parent rematerialization below. That historical prerequisite must reproduce all preregistered parent hashes byte for byte. The v3 builder and evaluator receive no qrels or index argument and never rerun retrieval.

The v3 design/audit split is inside the old 3,961-query inner-train development role. The 851-query audit partition is held out only from v3 design; it is not an untouched or independent test set. The 803-query future method holdout remains identity-only.

In [ ]:
builder = [
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/build_msmarco_rars_v2_boundary_bundles.py'),
    '--inner-only',
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--index', str(INDEX),
    '--qrels', str(CACHE / 'qrels_subset.json'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--cache-root', str(PARENT_CANDIDATE_CACHE),
    '--pca-config', str(TRAIN_REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
    '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
    '--pca-scales', str(PCA / 'sidecars/scales_pca_rank16.float32.npy'),
    '--pca-codes', str(PCA / 'sidecars/codes_pca_rank16.int8.memmap'),
    '--rars-config', str(CLEAN / 'selected_config.json'),
    '--rars-basis', str(CLEAN / 'bases/score_error_weighted_rank16.npy'),
    '--rars-scales', str(CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy'),
    '--rars-codes', str(CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap'),
    '--output-root', str(PARENT_BUNDLES),
    '--residual-batch-size', '20000',
]
subprocess.run(builder, check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)
bundle_summary = json.loads((PARENT_BUNDLES / 'bundle_build_summary.json').read_text())
assert bundle_summary['outer_validation_built'] is False
assert set(bundle_summary['roles']) == {'inner_train', 'inner_validation'}

subprocess.run([
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/freeze_rars_v2_2_inner_bundles.py'),
    '--bundle-root', str(PARENT_BUNDLES),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--outer-validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--clean-test-split', str(TRAIN_REPO / 'splits/msmarco_rars_test_split.json'),
    '--source-commit', TRAINING_COMMIT,
], check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)

parent = protocol['parent_lineage']
parent_hashes = {
    'parent_inner_train_manifest_sha256':
        sha256_file(PARENT_BUNDLES / 'inner_train/v2_2_manifest.json'),
    'parent_inner_train_source_manifest_sha256':
        sha256_file(PARENT_BUNDLES / 'inner_train/manifest.json'),
    'parent_inner_train_query_manifest_sha256':
        sha256_file(PARENT_BUNDLES / 'inner_train/query_manifest.json'),
    'closed_inner_validation_query_manifest_sha256':
        sha256_file(PARENT_BUNDLES / 'inner_validation/query_manifest.json'),
    'parent_v2_2_split_audit_sha256':
        sha256_file(PARENT_BUNDLES / 'v2_2_split_audit.json'),
}
for key, actual in parent_hashes.items():
    assert actual == parent[key], (key, actual, parent[key])
print(json.dumps({'status': 'EXACT_V2_2_PARENT_REMATERIALIZED', **parent_hashes}, indent=2))


In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON,
    str(V3_REPO / 'scripts/build_msmarco_rars_v3_oracle_bundles.py'),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--output-root', str(V3_BUNDLES),
    '--protocol', str(PROTOCOL_PATH),
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--n-docs', '1000000',
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)

candidate_summary = json.loads(
    (V3_BUNDLES / 'v3_oracle_bundle_freeze_summary.json').read_text()
)
assert candidate_summary['status'] == 'V3_QRELS_FREE_CANDIDATE_BUNDLES_FROZEN'
assert candidate_summary['parent_candidate_payloads_hash_verified'] is True
assert candidate_summary['parent_label_payload_bytes_read'] is False
assert candidate_summary['qrels_opened_or_parsed'] is False
assert candidate_summary['faiss_imported_or_search_performed'] is False
assert candidate_summary['pca_fit_or_score_recomputation_performed'] is False

ROLE_LABEL_FILES = (
    'candidate_relevance.uint8.npy',
    'relevant_counts.int32.npy',
    'v3_role_labels_started.json',
    'v3_role_labels_manifest.json',
)
for role in ('oracle_design', 'oracle_audit'):
    assert not any((V3_BUNDLES / role / name).exists() for name in ROLE_LABEL_FILES)
future_files = {path.name for path in (V3_BUNDLES / 'future_method_holdout').iterdir()}
assert future_files == {'query_manifest.json', 'v3_identity_manifest.json'}
print('Qrels-free design/audit candidates and future identity are frozen.')


In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON,
    str(V3_REPO / 'scripts/materialize_rars_v3_role_labels.py'),
    '--bundle-root', str(V3_BUNDLES),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--role', 'oracle_design',
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--protocol', str(PROTOCOL_PATH),
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)

design_label_manifest = V3_BUNDLES / 'oracle_design/v3_role_labels_manifest.json'
design_labels = json.loads(design_label_manifest.read_text())
assert design_labels['status'] == 'ROLE_LABELS_MATERIALIZED_FROM_FROZEN_PARENT'
assert design_labels['role_id'] == 'oracle_design'
assert not any(
    (V3_BUNDLES / 'oracle_audit' / name).exists() for name in ROLE_LABEL_FILES
)
print('Design labels materialized; audit labels remain absent.')


In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON,
    str(V3_REPO / 'scripts/evaluate_rars_v3_oracle_first_feasibility.py'),
    '--phase', 'design',
    '--bundle-root', str(V3_BUNDLES),
    '--design-bundle', str(V3_BUNDLES / 'oracle_design'),
    '--design-label-manifest', str(design_label_manifest),
    '--audit-bundle', str(V3_BUNDLES / 'oracle_audit'),
    '--protocol', str(PROTOCOL_PATH),
    '--output-dir', str(OUTPUT),
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--reuse-complete',
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)

design_freeze = json.loads((OUTPUT / 'design_freeze.json').read_text())
assert design_freeze['status'] == 'DESIGN_ARTIFACTS_FROZEN_BEFORE_AUDIT_LOAD'
assert design_freeze['source_commit'] == V3_IMPLEMENTATION_COMMIT
assert design_freeze['audit_bundle_loaded_before_this_freeze'] is False
assert design_freeze['audit_role_labels_materialized_before_this_freeze'] is False
assert design_freeze['future_method_holdout_accessed'] is False
assert not any(
    (V3_BUNDLES / 'oracle_audit' / name).exists() for name in ROLE_LABEL_FILES
)
print(json.dumps({
    'status': design_freeze['status'],
    'selected_primary_comparator': design_freeze['selected_primary_comparator'],
    'design_fold_gains': design_freeze['design_fold_gains'],
}, indent=2))


## Results

The next cell is the first audit-outcome access. The evaluator first verifies the complete design freeze, then writes the audit-start marker, materializes the exact audit label slice internally, loads audit arrays, and evaluates the full registered curve. Do not interrupt the cell or edit the output directory.

In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON,
    str(V3_REPO / 'scripts/evaluate_rars_v3_oracle_first_feasibility.py'),
    '--phase', 'audit',
    '--bundle-root', str(V3_BUNDLES),
    '--audit-bundle', str(V3_BUNDLES / 'oracle_audit'),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--protocol', str(PROTOCOL_PATH),
    '--output-dir', str(OUTPUT),
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--reuse-complete',
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)
print('Audit phase completed or an exact prior complete run was verified.')


In [ ]:
complete_path = OUTPUT / 'oracle_complete.json'
summary_path = OUTPUT / 'oracle_summary.json'
complete = json.loads(complete_path.read_text())
summary = json.loads(summary_path.read_text())
assert complete['status'] == 'ORACLE_COMPLETE'
assert summary['status'] == 'ORACLE_COMPLETE'
assert complete['source_commit'] == V3_IMPLEMENTATION_COMMIT
assert summary['source_commit'] == V3_IMPLEMENTATION_COMMIT
assert complete['run_fingerprint'] == summary['run_fingerprint']

root = OUTPUT.resolve()
for relative_name, record in complete['outputs'].items():
    relative = Path(relative_name)
    assert not relative.is_absolute() and '..' not in relative.parts, relative
    path = (OUTPUT / relative).resolve()
    assert root in path.parents, path
    verify_record(path, record)
verify_record(OUTPUT / 'design_freeze.json', complete['design_freeze'])

report = {
    'formal_decision': summary['formal_decision'],
    'evidence_status': summary['evidence_status'],
    'selected_primary_comparator': summary['selected_primary_comparator'],
    'mean_recall_at_10': {
        key: summary['mean_recall_at_10'][key]
        for key in (
            'base', 'primary_comparator', 'Exact40', 'Exact100',
            'Oracle0', 'Oracle8', 'Oracle16', 'Oracle32',
        )
    },
    'oracle_budget_curve': summary['oracle_budget_curve'],
    'oracle0_contract': summary['oracle0_contract'],
    'bootstrap_oracle16_vs_primary_comparator': summary['bootstrap'],
    'comparator_relative_recovery': {
        key: summary['counterfactual_recovery']['comparator_relative'][key]
        for key in ('Oracle8', 'Oracle16')
    },
    'gate_checks': summary['gate']['checks'],
    'output_dir': str(OUTPUT),
}
print(json.dumps(report, indent=2, allow_nan=False))


## Takeaways

Interpret the formal decision literally:

- KILL_NO_SCORE_HEADROOM: the residual-score reference cannot clear the preregistered comparator headroom; stop this method line.
- STOP_NO_HEADROOM: some oracle effect, support, concentration, harm, recovery, fold, or uncertainty gate failed; do not train an allocator.
- GO_TO_STATIC_STORAGE_ORACLE: accessed-byte headroom is broad enough to justify a new, separately frozen serialized-storage oracle protocol only.
- INVALID: do not interpret numerical outputs; repair provenance or execution and rerun from a clean local materialization without deleting the durable record.

Return \`oracle_summary.json\`, \`oracle_complete.json\`, and the printed report for review. No result from this notebook is an independent confirmation or a deployable storage-compression claim.